# Part 4: Work IQ로 업무 맥락 추가하기

받은 편지함을 열어보니 긴급 이메일 세 통이 와 있습니다. 시애틀 매장에서 프로페셔널 클로 해머(Professional Claw Hammer)가 품절되었고, 고객들이 불만을 제기하고 있으며, 동료들이 이전이나 긴급 재입고를 조율해 달라고 요청하고 있습니다. 이메일 스레드의 내용을 파악하고, 긴급 조달 에스컬레이션에 대해 회사 핸드북이 무엇을 명시하는지 알아내야 합니다.

**🎯 미션**
- 실제 Microsoft 365 데이터(이메일, Teams, 캘린더)를 질의할 수 있는 **Work IQ 지식 소스(Work IQ knowledge source)** 추가하기
- 3개 소스로 구성된 지식 베이스(HR 문서 + 건강 문서 + Work IQ) 구축하기
- 하나의 하위 답변은 받은 편지함에서, 다른 하나는 회사 정책에서 나오는 질문하기

## 빌딩 블록

Part 1~3에서 지식 베이스는 문서와 구조화된 데이터를 질의했습니다. 이제 개인 업무 맥락에 연결하는 소스를 추가합니다.

- **Work IQ 지식 소스(Work IQ Knowledge Source)**: Microsoft 365(Outlook, Teams, 캘린더, OneDrive)에 연결합니다. 지식 베이스가 런타임에 이를 질의하여 사용자 질문과 관련된 이메일, 채팅, 캘린더 일정을 표면화합니다.
- **위임 토큰(Delegated token)**: 이 노트북은 `InteractiveBrowserCredential`을 사용해 현재 사용자를 로그인시키고 Azure AI Search에 대한 위임 토큰을 획득합니다. 검색은 Work IQ 같은 보호된 소스를 질의할 때 그 위임된 사용자 ID를 사용합니다.

## 1단계: 환경 변수 설정

Azure AI Search 및 Azure OpenAI 리소스에 대한 구성을 로드합니다. 프로젝트 폴더의 `.env` 파일에 이 파트에 필요한 값이 들어 있습니다.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

print("Environment variables loaded")

## 2단계: 로그인 및 위임 토큰 획득

지식 베이스 검색이 Work IQ와 함께 여러분의 ID를 사용하려면, 이 테넌트의 Microsoft 365 데이터에 액세스 권한이 있는 로그인된 사용자의 OAuth 토큰이 필요합니다.

아래 셀을 실행하여 안내 사이드바의 자격 증명으로 Azure에 로그인하세요. 성공하면 Azure AI Search 범위에 대한 위임 토큰을 획득하고 검색 중 `query_source_authorization`으로 사용하도록 저장합니다.

In [ ]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=AZURE_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

## 3단계: 세 개의 지식 소스 만들기

이 지식 베이스를 위해 세 개의 지식 소스를 만듭니다. 앞에서 사용한 것과 동일한 두 개의 검색 인덱스 소스에 더해, Work IQ 지식 소스 하나입니다.

* `hrdocs-knowledge-source`: `hrdocs` 검색 인덱스를 가리킴
* `healthdocs-knowledge-source`: `healthdocs` 검색 인덱스를 가리킴
* `workiq-knowledge-source`: 업무 맥락을 위해 Work IQ를 가리킴

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Zava Work IQ knowledge source",
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)
print("Knowledge sources created")


## 4단계: 멀티 소스 + Work IQ 지식 베이스 만들기

지식 베이스는 지식 소스를 LLM 및 검색 동작 방식에 대한 지침과 결합합니다.

이 노트북에서 지식 베이스는 연결된 모델이 질문에 직접 답할 수 있도록 `answerSynthesis` `outputMode`를 사용합니다. 또한 모델이 질의 계획과 소스 선택을 도울 수 있도록 `low` `retrievalReasoningEffort`를 사용합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-workiq-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining indexed company documents and Work IQ",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for workplace context (emails, chats, events, meetings, etc). Use the search indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## 5단계: 지식 베이스에 질의

두 부분으로 된 질문을 합니다. 한쪽은 Work IQ에서 가져온 실제 받은 편지함 맥락이 필요하고, 다른 한쪽은 HR 인덱스의 회사 정책이 필요합니다.

* _"시애틀의 프로페셔널 클로 해머 재고 문제에 대한 최근 이메일이 무엇인가요?"_ → **Work IQ** 에서 와야 함
* _"재고 전략과 예산 감독을 담당하는 Zava 직무는 무엇인가요?"_ → `hrdocs`에서 와야 함

> ⏳ Work IQ가 검색 지연 시간을 늘리므로 응답을 1~2분 정도 기다려야 합니다. 검색 요청은 `max_runtime_in_seconds`를 지정하여 지식 베이스가 소스로부터 결과를 기다리는 시간을 늘립니다.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = ("프로페셔널 클로 해머에 대한 최근 업무 이메일을 확인해 주세요. "
            "동료들이 어떤 이야기를 하고 있고 어떤 조치가 요청되었는지 요약해 주세요. " 
            "재고 전략과 예산 감독을 담당하는 Zava 직무는 무엇인가요?")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))


### 활동 로그 검토

이 활동 로그에서는 Work IQ에 대한 호출에 해당하는 "workIQ" 단계와, 각 검색 인덱스로 보낸 질의에 해당하는 "searchIndex" 단계를 볼 수 있습니다.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### 참조 검토

Work IQ 지식 소스의 경우 참조는 `type: "workIQ"`를 가집니다. `sourceData`에는 Work IQ 에이전트가 합성한 답변이 담긴 `extracts`가 포함되며, 각 참조에는 참조된 이메일, 문서 등으로 연결되는 URL이 담긴 `attributions`가 포함됩니다.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 소스 헌트

위에 출력된 참조를 살펴보세요. 다음을 식별할 수 있나요?

1. 어떤 지식 소스가 **이메일 스레드 요약** 질문에 답했나요?
2. 어떤 지식 소스가 **직무** 질문에 답했나요?

## ✅ 미션 완료

**무엇을 만들었나:**

* ✓ **Work IQ 지식 소스(Work IQ Knowledge Source)**: 실제 Microsoft 365 데이터를 지식 베이스에 연결하여, 지식 베이스가 질의 시점에 이메일, Teams 메시지, 캘린더 맥락을 검색할 수 있게 하는 소스.
* ✓ **3개 소스 지식 베이스**: HR 문서, 건강 문서, 라이브 M365 업무 데이터에 걸친 단일 지식 베이스로, 에이전트가 각 하위 질문을 적절한 소스로 라우팅함.
* ✓ **받은 편지함/정책 하이브리드 검색**: Work IQ가 시애틀 재고 부족에 대한 이메일 스레드를 표면화했고, HR 인덱스가 예산 관리 질문에 답했음.

➡️ [Part 5: Work IQ와 Fabric IQ 결합하기](part5-work-iq-fabric-iq-to-kb.ipynb)로 계속 진행하세요.